In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from etl.common import init_spark3

import sys
import time
import importlib
import pandas as pd

BASE_DIR = Path("/apps/jupyter/users/nguyetnguyen/features")

sys.path[:0] = [str(path) for path in [
                    BASE_DIR / "common" / "src",
                    BASE_DIR / "device_refactored",
                ]
]

from common.merge_features import merge_features_from_sample_path
from common.utils import dir_ls
from common.agg_columns import (
    build_groupby_window_query,
    build_aggregate_columns,
    get_lxw_windows,
    get_lxw_overlap_windows,
    get_feature_columns,
)
from src.device_helpers import (
    register_temp_view,
)

import configs.configs as config

spark = init_spark3.setup(
    job_cfg={
        'executor.instances': 8,
        'executor.cores': 8,
        'executor.memory': '20g',
    },
    script_name="build_device_tac_features"
)

In [ ]:
def df_pn_imei_tac_raw_weekly():
    return (
        spark.read.parquet(config.IMEI_WEEKLY_HC_SAMPLE_PATH)
        .filter(
            (F.col("date") > config.START_DATE_TAC_FEATURES)
            & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
    )

def df_pn_last_activated_date():
    return (
        spark.read.parquet(config.LAST_ACTIVATED_PATH)
        .filter(
            (F.col("date") > config.START_DATE_LAST_ACTIVATED_DATE)
            & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
        .groupBy("phone_number")
        .agg(F.max("last_activated_date").alias("last_activated_date"))
    )

def df_pn_imei_tac_weekly():
    return (
        df_pn_imei_tac_raw_weekly()
        .join(df_pn_last_activated_date(), "phone_number")
        .where("date > last_activated_date")
        .drop("last_activated_date")
    )

def df_tac_shared_pn_counts_lxw():
    source = register_temp_view(df_pn_imei_tac_weekly(), "df_pn_imei_tac_weekly")
    stats = [("count(distinct phone_number)", "device_tac_distinct_pn_count")]
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["tac"],
        source=source,
    )
    return spark.sql(query)

def df_pn_by_tac_event_stats_lxw():
    source = register_temp_view(df_pn_imei_tac_weekly(), "df_pn_imei_tac_weekly")
    stats = [
        ("count(phone_number)", "device_pn_tac_events_count"),
        ("count(distinct date)", "device_pn_tac_distinct_weeks_count"),
        ("sum(device_num_day_l1w)", "device_pn_tac_distinct_days_count"),
    ]
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["phone_number", "tac"],
        source=source,
    )
    return spark.sql(query)

def df_pn_tac_distinct_counts_lxw():
    source = register_temp_view(df_pn_imei_tac_weekly(), "df_pn_imei_tac_weekly")
    stats = [
        ("count(distinct tac)", "device_pn_tac_distinct_count"),
    ]
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["phone_number"],
        source=source,
    )
    return spark.sql(query)

def df_pn_tac_overlap_features_lxw():
    def df_pn_tac_sets_lxw():
        source = register_temp_view(df_pn_imei_tac_weekly(), "df_pn_imei_tac_weekly")
        windows = get_lxw_overlap_windows(config.WINDOW_WEEKS[:-1])
        stats = [("collect_set(tac)", "device_tac_set")]
        query, _ = build_groupby_window_query(
            agg_exprs=stats, windows=windows, snapshot_date=config.SNAPSHOT_DATE_STR,
            group_by=["phone_number"], source=source,
        )
        return spark.sql(query)

    sets_df = df_pn_tac_sets_lxw()
    source = register_temp_view(sets_df, "df_pn_tac_sets_lxw")
    prev_columns = get_feature_columns(sets_df, exclude=["phone_number"])
    columns, _ = build_aggregate_columns(prev_columns, ["intersection_ratio", "except_ratio"])

    query = f"select phone_number, {', '.join(columns)} from {source}"
    return spark.sql(query)

def df_pn_tac_distribution_lxw():
    df_join = (
        df_pn_by_tac_event_stats_lxw()
        .join(df_tac_shared_pn_counts_lxw(), on="tac", how="left")
    )

    source = register_temp_view(df_join, "df_join")
    prev_columns = get_feature_columns(df_join, exclude=["phone_number", "tac"])

    stat_config = [
        {"func": "min", "exclude": ['device_pn_tac_events_count', 'device_tac_distinct_pn_count']},
        {"func": "max"},
        {"func": "avg", "exclude": ['device_tac_distinct_pn_count']},
        {"func": "std"},
        {"func": "skewness"},
        {"func": "kurtosis"},
    ]
    columns, _ = build_aggregate_columns(prev_columns, stat_config)

    query = f"select phone_number, {', '.join(columns)} from {source} group by phone_number"
    return spark.sql(query)

def df_pn_tac_entropy_features_lxw():
    df_join = (
        df_pn_by_tac_event_stats_lxw()
        .join(df_tac_shared_pn_counts_lxw(), on="tac", how="left")
    )

    source = register_temp_view(df_join, "df_join")
    prev_columns = get_feature_columns(df_join, exclude=["phone_number", "tac"])
    columns, sub_columns = build_aggregate_columns(prev_columns, ["entropy"])

    query = f"""
select phone_number, {', '.join(columns)}
from (select phone_number, {', '.join(sub_columns)} from {source})
group by phone_number
"""
    return spark.sql(query)

def df_pn_tac_features_lxw():
    df = (
        df_pn_tac_distribution_lxw()
        .join(df_pn_tac_entropy_features_lxw(), on="phone_number", how="outer")
        .join(df_pn_tac_distinct_counts_lxw(), on="phone_number", how="outer")
        .join(df_pn_tac_overlap_features_lxw(), on="phone_number", how="outer")
    )
    return df.select([
        F.when(F.isnan(c), None).otherwise(F.col(c)).alias(c)
        for c in df.columns
    ])

In [ ]:
for snapshot_date in pd.date_range('2024-05-19', '2024-10-31', freq='W-SUN'):
    start_time = time.time()
    snapshot_date_str = snapshot_date.strftime('%Y-%m-%d')
    
    config.SNAPSHOT_DATE_STR = snapshot_date_str
    config.SNAPSHOT_DATE = datetime.strptime(config.SNAPSHOT_DATE_STR, config.DATE_FORMAT)
    config.START_DATE_TAC_FEATURES = (config.SNAPSHOT_DATE - timedelta(days=104*7)).strftime(config.DATE_FORMAT)
    config.START_DATE_LAST_ACTIVATED_DATE = (config.SNAPSHOT_DATE - timedelta(days=2*7)).strftime(config.DATE_FORMAT)
    
    print(datetime.now(), config.SNAPSHOT_DATE_STR)
    
#    df_pn_tac_features_lxw().write.mode('overwrite').parquet(f'{config.DEVICE_TAC_FEATURES_HC_SAMPLE_PATH}/date={config.SNAPSHOT_DATE_STR}')
    
    end_time = time.time()
    
    print(f'Done at: {datetime.now()} during {end_time - start_time}')
    print('-'*50)